In [ ]:
# Model
from pprint import pprint

from langchain_ollama import ChatOllama

MODEL = "gemma3:1b"  # Small: fast iterations and development
# MODEL = "gemma3:4b"   # Medium: balanced quality and speed
# MODEL = "gemma3:12b"  # Large: highest quality final outputs

llm = ChatOllama(
    base_url="http://host.docker.internal:11434",
    keep_alive="12m",
    model=MODEL,
    temperature=0.24,
)
pprint(llm)

In [ ]:
# Run Model
from pprint import pprint

hello_world = llm.invoke("Hello, world!")
hello_world.pretty_print()
pprint(hello_world.model_dump(exclude={"content"}))

In [ ]:
# Search
import trafilatura
from langchain_community.utilities import SearxSearchWrapper

from src.prompts import search_query_prompt_template
from src.subjects import TAROT_CARDS

searxng_wrapper = SearxSearchWrapper(searx_host="http://searxng:8080")


def search(query: str) -> str:
    """Search using SearxNG and fetch full text content of the top results via trafilatura."""
    searxng_results = searxng_wrapper.results(query, num_results=12)
    trafilatura_results = []
    for i, result in enumerate(searxng_results, start=1):
        url = result.get("link", "")
        title = result.get("title", "").strip()
        text = trafilatura.extract(
            trafilatura.fetch_url(url),
            include_comments=False,
            include_tables=True,
            no_fallback=False,
        )
        heading = title if title else url
        trafilatura_results.append(
            f"## Search Result {i}: {heading}\n\n**Source:** [{url}]({url})\n\n{text}"
        )

    if not trafilatura_results:
        raise ValueError(f"No content could be extracted for query: {query}")

    return "\n\n---\n\n".join(trafilatura_results)


subject = TAROT_CARDS[0]
search_results = search(
    search_query_prompt_template.format(
        subject_name=subject.name,
        category_name=subject.category.name,
    )
)
pprint(search_results)

In [ ]:
# One-Shot Document Ingestion
import re
from pathlib import Path

import inflect

from src.prompts import ingest_document_prompt_template, search_query_prompt_template
from src.subjects import TAROT_CARDS, Subject

p = inflect.engine()


def strip_markdown_fences(content: str) -> str:
    """Strip leading/trailing markdown code fences that small LLMs sometimes emit."""
    content = content.strip()
    content = re.sub(r"^```(?:markdown)?\s*\n", "", content)
    content = re.sub(r"\n```\s*$", "", content)
    return content.strip()


def write_document(subject: Subject, content: str) -> None:
    output_directory = Path(f"../../output/{subject.category.slug}")
    output_directory.mkdir(parents=True, exist_ok=True)
    output_filename = f"{subject.order}-{subject.slug}.md"
    output_path = output_directory / output_filename
    output_path.write_text(content, encoding="utf-8")
    print(f"\nWritten to: {output_path.resolve()}\n")


def ingest_document(subject: Subject) -> None:
    search_results = search(
        search_query_prompt_template.format(
            subject_name=subject.name,
            category_name=subject.category.name,
        )
    )

    messages = ingest_document_prompt_template.format_messages(
        subject_name=subject.name,
        category_name=subject.category.name,
        category_name_plural=p.plural(subject.category.name),
        search_results=search_results,
    )
    for message in messages:
        message.pretty_print()

    agent_message = llm.invoke(messages)
    agent_message.pretty_print()

    content = strip_markdown_fences(str(agent_message.content))
    write_document(subject, content)


# for subject in TAROT_CARDS[28:29]:
#     ingest_document(subject)

In [ ]:
# Chain of Thought Graph
# Two-step CoT: Node 1 (analyze) compresses raw search results into a research brief;
# Node 2 (write) produces the final document from the distilled analysis.
# This reduces context pressure on small models and yields fewer hallucinated correspondences.
from typing import NotRequired, TypedDict

import inflect
from IPython.display import Image, display
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph import END, START, StateGraph

from src.prompts import analysis_prompt_template, write_document_prompt_template
from src.subjects import TAROT_CARDS, Subject

p = inflect.engine()


class IngestionState(TypedDict):
    subject_name: str
    category_name: str
    category_name_plural: str
    search_results: str
    analysis: NotRequired[str]
    document: NotRequired[str]


_parser = StrOutputParser()


def analyze(state: IngestionState) -> dict[str, str]:
    messages = analysis_prompt_template.format_messages(
        subject_name=state["subject_name"],
        category_name=state["category_name"],
        search_results=state["search_results"],
    )
    return {"analysis": _parser.invoke(llm.invoke(messages))}


def write(state: IngestionState) -> dict[str, str]:
    messages = write_document_prompt_template.format_messages(
        subject_name=state["subject_name"],
        category_name=state["category_name"],
        category_name_plural=state["category_name_plural"],
        analysis=state.get("analysis", ""),
    )
    return {"document": _parser.invoke(llm.invoke(messages))}


graph: StateGraph = StateGraph(IngestionState)
graph.add_node("analyze", analyze)
graph.add_node("write", write)
graph.add_edge(START, "analyze")
graph.add_edge("analyze", "write")
graph.add_edge("write", END)
ingest_graph = graph.compile()
display(Image(ingest_graph.get_graph().draw_mermaid_png()))


def ingest_document_chain_of_thought(subject: Subject) -> None:
    search_results = search(
        search_query_prompt_template.format(
            subject_name=subject.name,
            category_name=subject.category.name,
        )
    )

    state = ingest_graph.invoke(
        {
            "subject_name": subject.name,
            "category_name": subject.category.name,
            "category_name_plural": p.plural(subject.category.name),
            "search_results": search_results,
        }
    )

    print("=== ANALYSIS ===")
    print(state["analysis"])

    content = state["document"]
    write_document(subject, content)


for subject in TAROT_CARDS[28:29]:
    ingest_document_chain_of_thought(subject)